In [1]:
import pandas as pd
import numpy as np

In [ ]:
# read rich fields data
rich = pd.read_csv('',
                   dtype={
                       'cellphone_number' : 'str',
                       'id_number': 'str'
                   })

In [ ]:
data = (
    pd.read_excel(
        io='',
        usecols=range(2, 8),
        skiprows=1
    )
    .assign(
        umuzi_email = lambda df: df["Comma separated list of emails"].str.split(',')
    )
    .drop(
        columns=["Comma separated list of emails", "If useful for person compiing the data"]
    )
    .explode(column="umuzi_email", ignore_index=True)
    .assign(
        umuzi_email = lambda df: df["umuzi_email"].str.strip(),
        cohort = lambda df: df["cohort"].str.strip()
    )
    .rename(
        columns={
            "type of support service used": "service_used",
            " - type of connection session": "service_name",
            "format DD/MM/YYYY - if only month and year known, put DD as 01": "date_service_accessed",
            "cohort": "cohort_name"
        }
    )
    .assign(
        cohort_name = lambda df: df['cohort_name'].map({
            "Westbank": "SL-PM-Wesbank-Nov-25",
            "C50": "XH-AUXUI-Aug-25",
            "C51": "SL-FWD-Feb26",
            "A5": "XH-AWD-BBD-Feb-26",
            'WesBank': "SL-PM-Wesbank-Nov-25"
        })
    )
)

In [4]:
data.head()

,cohort_name,service_used,service_name,date_service_accessed,umuzi_email
0,XH-AUXUI-Aug-25,Stand-Up,LX Coach,2026-02-03,ann.ntabeni@umuzi.org
1,XH-AUXUI-Aug-25,Stand-Up,LX Coach,2026-02-03,boipelo.kekana@umuzi.org
2,XH-AUXUI-Aug-25,Stand-Up,LX Coach,2026-02-03,khanyisile.kabini@umuzi.org
3,XH-AUXUI-Aug-25,Stand-Up,LX Coach,2026-02-03,lesego.mbenge@umuzi.org
4,XH-AUXUI-Aug-25,Stand-Up,LX Coach,2026-02-03,malethola.raditsela@umuzi.org


In [5]:
premerge_rows = data.shape[0]

In [6]:
# merge with main data
data = data.merge(
            right=rich,
            on='umuzi_email',
            how='left'
        )

In [7]:
# make sure no new rows are introduced due to duplicates
assert premerge_rows == data.shape[0], f"Row mismatch; expected shape:{premerge_rows, data.shape[1]}, instead got {data.shape}"

In [8]:
# Not viable to be reported
data.loc[data['learner_id'].isna()]['umuzi_email'].sort_values().unique().tolist()

[]

In [9]:
data[data['date_of_birth'].isna()]['umuzi_email'].sort_values().unique().tolist()

[]

In [10]:
data.dropna(subset=['learner_id', 'date_of_birth'], inplace=True)

> ENRICHMENT TIME!

In [11]:
data.isna().sum()

cohort_name                             0
service_used                            0
service_name                            0
date_service_accessed                   0
umuzi_email                             0
learner_id                              0
application_id                          2
application_date                        2
month_of_registration                   2
email                                   0
first_name                              0
last_name                               0
cellphone_number                        1
id_number                               2
race                                    2
gender                                  2
date_of_birth                           0
has_disability_or_differently_abled     2
nearest_metro                           4
province                               39
residential_area_type                  11
dtype: int64

In [12]:
data['learner_id'] = data['learner_id'].astype('int')
data['id_number'] = data['id_number'].str.rjust(13, '0')

In [13]:
data['age_service_accessed'] = (
    pd.to_datetime(data['date_service_accessed'])- pd.to_datetime(data['date_of_birth'])
).dt.days // 365

# add age bins
data['age_range'] = pd.cut(
    x=data['age_service_accessed'],
    bins=[-np.inf, 17, 25, 35, np.inf],
    labels=['17 and below', '18-25', '26-35', '36+']
)

In [14]:
data['month_of_service_accessed'] = pd.to_datetime(data['date_service_accessed']).dt.month_name()

data['month_of_service_accessed'] = data['month_of_service_accessed'] + ' 2025'

In [15]:
data = data[[
    'learner_id', 'application_id', 'umuzi_email',
    'first_name', 'last_name', 'cohort_name', 'cellphone_number',
    'id_number', 'gender', 'date_of_birth', 'age_service_accessed', 'age_range', 'race',
    'residential_area_type', 'has_disability_or_differently_abled',
    'application_date', 'nearest_metro', 'province', 'date_service_accessed',
    'month_of_service_accessed', 'service_used', 'service_name'
]]

In [ ]:
# data.to_csv('../Sink Datasets/indicator_7_data.csv', index=False)

In [ ]:
# data.to_csv('../Sink Datasets/indicator_7_lxcheckins.csv', index=False)